# Final project: Gait, injury, and the trouble with speed

**Question.** Can you tell an injured runner from a healthy one from their
walking gait? If a model seems to work, what is it really using?

You have gait features for ~1,600 people, and the raw joint-angle curves for
almost all of them. Most of these people have a clinically diagnosed (but
currently pain-free) running injury. Build an honest machine-learning pipeline that
tries to separate the injured and healthy groups. Then reason about what drives
your model.

This is a **methods** project. Your grade does not depend on high accuracy. The
injury signal is subtle and the classes are imbalanced. Your grade depends on doing
the analysis correctly and reporting it honestly:

* a **train/test split first**, cross-validation (CV) **inside the training data only**,
  and **one** look at the test set at the end,
* honest baselines,
* handling the class imbalance,
* testing whether speed or demographics explain your result,
* using both the gait features **and** the raw joint-angle signals,
* clear conclusions.

**How this works.** This is the suggested project for the course. Work in a group of
2 to 4 people. You do **not** submit this notebook. You use it to do the analysis,
and then you write a **report** and give a **10-minute presentation** about your
work. See *Deliverables* below.

**Before you start,** read `DATA_DICTIONARY.md` (the columns) and `CURVES_README.md`
(the raw curves). Keep this notebook in the same folder as the data files.

## The rules of the protocol

1. Split the data into train and test **before** you explore or model (Milestone 0).
2. Use only the **train** set for exploring, comparing models, choosing features,
   and tuning. Use cross-validation (CV) **inside** train to score ideas.
3. Put every preprocessing step (scaling, PCA) **inside a `Pipeline`**, so it is
   fit on the training folds only.
4. Do not look at the test labels or test scores until Milestone 7.
5. Score the test set **once**, with the model you chose using CV. If you change
   your model after you see the test score, the test set is no longer honest. If
   this happens, say so in your report.
6. Use a fixed `random_state` for every split, CV object and model, so the notebook
   gives the same numbers each time it runs.

## Deliverables

You submit two things. You do **not** submit this notebook, but keep it: every number,
table and figure in your report and slides must come from it, and it must run from
top to bottom (**Kernel → Restart & Run All**).

### 1. Written report
* Content: a short description of the problem, the data, your methods, a summary of
  your results, and a final discussion.
* Format: Arial 11 pt, **maximum 10 pages**, APA reference style.
* Cite the dataset paper (Ferber et al., 2024; see `README.md`).

### 2. Presentation
* **10 minutes**, PowerPoint, presented to the class. You lose 1 point for each
  minute over or under 10 minutes.
* Include: Background, Methods, Results, Discussion.
* Do not repeat material that was taught in class. Focus on what is new in your
  work, and on why your results matter for the study of human movement.

### Scope by group size
Larger groups are expected to do more:

| Group size | Required |
| --- | --- |
| 2 people | Milestones 0 to 7, Milestone 8 is optional |
| 3 people | Milestones 0 to 7, plus Milestone 8 **or** one stretch goal |
| 4 people | Milestones 0 to 7, plus Milestone 8 **and** one stretch goal |

**Software:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`. Milestone 8 also
needs `torch` (PyTorch).

**Generative AI:** follow the course policy. You may use AI to explain code, find
bugs and write documentation. The analysis, the report and the slides must be your
own work.

Suggested outlines for the report and the slides, and the grading rubrics, are at the
end of this notebook.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FEATURES = ['cadence', 'stride_length', 'stance_time', 'vertical_oscillation',
            'peak_knee_flexion', 'peak_hip_extension', 'peak_ankle_dorsiflexion',
            'peak_hip_adduction', 'peak_pelvic_drop', 'peak_eversion',
            'sym_knee_flexion', 'sym_stance_time', 'stride_time_cv']
MODEL_COLS = FEATURES + ['speed', 'age', 'sex', 'height_cm', 'mass_kg', 'bmi']

walk = pd.read_csv('features_walk.csv')
# keep the two clear groups; drop rows missing a modelling value.
# (injury_joint / specific_injury are blank for healthy people: do not drop on those)
walk = walk[walk['injury_status'].isin(['healthy', 'injured'])].dropna(subset=MODEL_COLS).reset_index(drop=True)
walk['injured'] = (walk['injury_status'] == 'injured').astype(int)   # 1 = injured
walk['male'] = (walk['sex'] == 'Male').astype(int) #1 = male
print(walk.shape, walk['injury_status'].value_counts().to_dict())
walk.head()

---
## Milestone 0: Split the data (do this before anything else)

* Split `walk` into **train (80%)** and **test (20%)**. Call the two tables
  `train` and `test`. The rest of the notebook uses these names.
* Use a **stratified** split on `injured`, so both sets keep the same class ratio.
  (Hint: `sklearn.model_selection.train_test_split` with `stratify=` and a fixed
  `random_state`.)
* Print the share of injured people in `train` and in `test`. They must be almost
  the same.
* Make your CV object now (for example `StratifiedKFold(5, shuffle=True, random_state=0)`).
  Use it only on the train set.
* From here on, use `train`. Do not use `test` until Milestone 7.

Why one row per person makes a plain split valid: see `DATA_DICTIONARY.md`.

In [ ]:
# M0: split, then check the class ratio in train and test

---
## Milestone 1: Explore (train only)

* How many healthy and how many injured people are in `train`? Is it balanced?
  Write down the ratio. It affects which metric you can use.
* Summarise the features (for example with `train.describe()`). Plot the
  distributions of a few features by group (healthy vs injured) with pandas or
  matplotlib.
* **Compare walking speed between the groups.** Do the injured people walk at a
  different speed? Look at the mean **and** the spread. Keep this result. In
  Milestones 2 and 3 you will test whether speed explains your model.

In [ ]:
# M1: your exploration (train only)

*Notes for your report (Milestone 1):* What is the class ratio? Do the groups differ in
walking speed (mean and spread)?

---
## Milestone 2: Baselines first (CV on train)

Before you build a real model, find the score of an honest baseline. Use CV on the
train set. Score these three baselines:

* **Majority class:** a "model" that predicts that everyone is injured. Report its
  accuracy (this is the share of injured people). Its ROC AUC is 0.5, because it
  gives the same score to every person. (Optional: `sklearn.dummy.DummyClassifier`.)
* **Speed only:** a model that uses only the `speed` column.
* **Age and sex only:** a model that uses only `age` and `male`.

For the speed-only and the age-and-sex models:

* Use ROC AUC (`cross_val_score(..., scoring='roc_auc')`). Report the mean and the
  standard deviation over the folds.
* Try **two model families**: logistic regression (with a `StandardScaler` in a
  `Pipeline`) and a random forest.
* Use `class_weight='balanced'`, the same as in your later models, so that the
  comparison is fair.

Keep these numbers. Every later model must beat them.

In [ ]:
# M2: baselines, cross-validated on train

*Notes for your report (Milestone 2):* Give the baseline AUCs (mean ± SD). Which baseline is
strongest? Do the two model families agree? If not, why might that be?

---
## Milestone 3: Predict injury from gait (CV on train)

Build real classifiers with the gait features in the `FEATURES` list (13 columns:
temporal, kinematic peaks, symmetry and variability). Do **not** add `speed`, age,
sex or body size yet. Try at least two model families (for example logistic
regression and a random forest).

* Use **ROC AUC**, not accuracy. Handle the imbalance (for example
  `class_weight='balanced'` when you create the model).
* Put the scaler inside a `Pipeline`.
* **Tune** at least one setting with CV on train (for example `GridSearchCV` for
  `C` in logistic regression, or `max_depth` in a random forest). Do not tune on
  the test set.
* The two **symmetry** columns are signed (right minus left). The injured leg is
  the right leg in some people and the left leg in others, so the sign can cancel.
  Also try the **absolute value**, for example
  `train['abs_sym_knee_flexion'] = train['sym_knee_flexion'].abs()`. If you use
  new columns, make the same columns in `test` before Milestone 7.
* **The confound test:** add `speed` to the features. Then also add the body size
  columns `height_cm`, `mass_kg` and `bmi`. Does the AUC change? If gait still beats
  the speed-only baseline, the signal is not only speed.
* Report the mean and the spread across folds, not one number.
* At the end, **choose one final model** (feature set, model type and settings)
  and write down why. This is the model you will score on the test set.

In [ ]:
# M3: injury classification, tuning, and the speed-confound test (train only)

*Notes for your report (Milestone 3):* What is your best CV AUC? How does it compare to the
speed baseline? Which model did you choose, and why?

---
## Milestone 4: A task that works, and a trap (CV on train)

### A) Predict walking speed (regression)
To see your pipeline succeed, **predict walking speed** (`speed`) with a regression
model (not a classifier), for example `RandomForestRegressor`. Use CV on the train
set (`KFold`, because the target is a number) and report R² (`scoring='r2'`).

1. First use only `cadence` and `stride_length`. You will get a near-perfect R².
   Explain **why**. (Hint: what is cadence × stride length?)
2. Then use only the six joint-angle peak columns: `peak_knee_flexion`,
   `peak_hip_extension`, `peak_ankle_dorsiflexion`, `peak_hip_adduction`,
   `peak_pelvic_drop`, `peak_eversion`. This is the honest version.

In [ ]:
# M4.A: speed regression, with and without cadence + stride_length (train only)

### B) Classify sex
**Classify the walker's sex** (target column `male`) with logistic regression
(with a `StandardScaler` in a `Pipeline`). Use CV on the train set and report the
CV AUC.

1. First use the gait features in `FEATURES`.
2. Then use only the body measures `height_cm`, `mass_kg` and `bmi`.

In [ ]:
# M4.B: classify sex based on gait features or body measures (train only)

*Notes for your report (Milestone 4):* Why is speed almost perfectly predicted from cadence
and stride length? Why is sex much easier to predict than injury? What does the
body-measures result tell you about what a gait model can pick up?

---
## Milestone 5: The raw signals (CV on train)

So far you used features that someone else computed from the signals. Now use the
signals themselves. File: `curves_walk.npz`. For each person it has **12 curves of
101 points**: hip, knee and ankle, each in the sagittal and the frontal plane, for
the **right and the left** leg (6 + 6). A curve is the mean over all stance phases
(0-100% of stance). `CURVES_README.md` shows how to load the file.

1. Load the curves. Match them to your `train` and `test` tables **by `sub_id`**.
   Do not make a new split. A few people have no curves: drop them from both sets
   for this milestone.
2. Make three **views** of the curves (each is 6 curves x 101 points):
   * the **mean** of right and left, (R+L)/2,
   * the **signed asymmetry**, R-L,
   * the **absolute asymmetry**, |R-L|.

   Averaging alone would hide asymmetry. Think: the injured leg is the right leg
   in some people and the left leg in others. What does that do to the sign of R-L?
3. For each view, plot the mean curve of each group (healthy and injured), with
   the spread (±1 SD). What differs by eye?
4. Build a classifier for each view. Each view is 606 numbers per person, which is
   a lot for ~1,250 people. Start simple: flatten each view to 606 columns, reduce
   it with **PCA** (about 10 components), then use a classifier. Put the scaler
   **and** the PCA inside the `Pipeline`, so they are fit on the training folds only.
5. Score with CV on train: the mean view, each asymmetry view, and the mean +
   absolute asymmetry together. For a fair comparison, also run the CV of your
   Milestone 3 model on **the same people** (the train people who have curves).
   Then try your features **and** the curve PCA components together. (Hint: a
   `ColumnTransformer` lets you apply one PCA to each view.)
6. Choose **one** curves-only model to carry to the final test.

In [ ]:
# M5: load and plot the curves, then PCA + classifier (CV on train)

*Notes for your report (Milestone 5):* Do the raw curves predict injury better than the peak
features? Does asymmetry add anything? What does the answer say about where the
injury information is?

---
## Milestone 6: Interpret (train only)

* Which features did your injury model use most? (feature importances or
  coefficients, from a model fit on train)
* Does gait separate injured from healthy once speed is accounted for?
* Limitations: what makes the injury task hard here? What would you need to do
  better? (Think sample, labels, matching, and that people were pain-free at the
  test.)

In [ ]:
# M6: interpretation (train only)

*Notes for your report (Milestone 6):* Which features did your model use most? Does gait
separate injured from healthy once speed is accounted for? List the main
limitations of this data and say what you would need to do better.

---
## Milestone 7: Final test evaluation (one time only)

Now, and only now, use the test set.

1. Refit your **chosen** injury model (Milestone 3) on **all** the train data.
2. Score the test set **once**:
   * ROC AUC, with a bootstrap 95% interval. (Bootstrap: draw the test people
     with replacement 1,000 times, compute the AUC each time, and report the
     2.5th and 97.5th percentiles.)
   * A confusion matrix and the balanced accuracy, at a probability threshold of
     0.5 (`balanced_accuracy_score`).
   * The ROC curve (plot).
3. Score the speed-only baseline on the test set too (same model family as your
   chosen model, fit on train).
4. Score your speed-regression model (joint angles only, Milestone 4A) and your
   sex models (Milestone 4B) on the test set.
5. Score your chosen **raw-curve** model from Milestone 5 on the test set (people
   with curves). Next to it, score your features model on the same people.
6. Compare the CV scores (Milestones 3 and 5) with the test scores. Are they
   close? Why might they differ (small test set, tuning optimism)?

Do not go back and change any model after this step.

In [ ]:
# M7: final test evaluation

*Notes for your report (Milestone 7):* State your final test AUC and interval. Does your model
beat the speed-only baseline on **new** people? Do the raw curves help on new people?
What is your honest conclusion?

---
## Milestone 8 (ADDITIONAL): A small CNN on the raw curves

This step is extra. It makes a **more robust project**. It is optional for groups
of 2 people. Groups of 3 or 4 people: see *Scope by group size* in *Deliverables*.

Train a small 1-D convolutional neural network (CNN) on the 12 curves (right and
left, so it can learn asymmetry) to classify injured vs healthy. Then compare it
with your features model and your PCA-on-curves models from Milestone 5.

Rules (the same protocol):

* Use the same `train` / `test` people as before. Do not make a new split.
* Keep the network **small** (a few thousand parameters). You have only ~1,250
  training people. A big network will over-fit.
* Scale each curve using **training-part statistics only**.
* Handle the class imbalance (for example `pos_weight` in the loss).
* Choose the number of epochs with **CV inside train** (plot validation AUC against
  epoch). Do **not** use the test set for early stopping.
* Refit on all train people. Score the test set **once**. A single run is noisy, so
  average a few random seeds.

Starter network in PyTorch (you may change it):

```python
import torch, torch.nn as nn
net = nn.Sequential(
    nn.Conv1d(12, 16, 7, padding=3), nn.ReLU(), nn.MaxPool1d(2),
    nn.Conv1d(16, 32, 5, padding=2), nn.ReLU(), nn.MaxPool1d(2),
    nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(32, 1))
# input shape: (people, 12 curves, 101 points)
# your curve array is (people, 101, 12): use X.transpose(0, 2, 1)
```

Report a table: features model, PCA-on-curves models, features + curves, and CNN
(CV AUC and test AUC). Does the CNN help? Is any gap larger than the noise?

In [ ]:
# M8 (optional): CNN with CV inside train, then one test score

*Notes for your report (Milestone 8):* Compare the CNN with the other models. Is
the extra complexity worth it?

---
## Stretch goals

Optional for groups of 2 people. Groups of 3 or 4 people: see *Scope by group size*
in *Deliverables*.

* **Running vs walking.** Repeat the whole protocol on `features_run.csv` (new
  split, same rules). Does the injury show up more when people run?
* **Injury type.** The column `injury_joint` gives the injured joint. On the train
  set only, does your model separate some injury types (for example knee) from
  healthy people better than others?
* **Nested CV.** Estimate the tuning optimism: run the tuning inside an outer CV
  loop and compare the result with your test AUC.

---
## Suggested report outline (max 10 pages)

1. **Introduction (about 1 page).** Running injuries, why gait might show them, and
   the question of this project. Cite the literature (APA).
2. **Data (about 1 page).** The dataset (Ferber et al., 2024), who is in it, the
   groups, the class imbalance, the features and the curves you used, and the rows
   you dropped and why.
3. **Methods (2-3 pages).** The train/test split and cross-validation; the baselines;
   the models and how you tuned them; how you handled imbalance; the speed confound
   test; the curve views and PCA; the CNN (if you did it); the metrics (ROC AUC,
   bootstrap interval, balanced accuracy).
4. **Results (3-4 pages).** At least:
   * a table of CV AUC (mean ± SD) for the baselines and all main models,
   * a table of the final test results (AUC with 95% interval, balanced accuracy),
   * the ROC curve on the test set,
   * the mean curves by group (mean view and one asymmetry view),
   * the speed-regression and sex-classification results.
5. **Discussion (1-2 pages).** What your results mean for the study of human
   movement, whether gait carries an injury signal beyond speed, why the task is
   hard, the limitations, and what you would do next.
6. **References** (APA, not counted in the 10 pages).

## Suggested slide outline (10 minutes)

| Part | Slides | Content |
| --- | --- | --- |
| Background | 1-2 | the question, and why it matters for human movement |
| Methods | 2-3 | data, split and CV, models, curves |
| Results | 3-4 | baselines vs your best models, test result with interval, one key figure |
| Discussion | 1-2 | what the results mean, limitations, take-home message |

## How your work is graded

**Project report** (course rubric; see the syllabus for the full rubric):

| Course rubric item | Pts | What it means in this project |
| --- | --- | --- |
| Data preparation | 5 | clean two-group data; stratified train/test split made first; curves matched to the same split by `sub_id`; new columns (for example absolute symmetry) made the same way in train and test |
| Model selection | 5 | honest baselines; at least two model families; a clear reason for each final model (features and curves) |
| Model training | 5 | CV inside train only; scaling and PCA inside a `Pipeline`; tuning with CV; imbalance handled; test set scored once |
| Presentation of results | 15 | the report: data, preparation steps, models and results, clearly presented, with an honest discussion |

**Presentation** (course rubric):

* Duration: 1 point lost for each minute over or under 10 minutes.
* Clarity (50%): clear concepts and methods; focus on what is new.
* Use of audiovisual tools (25%): clear, concise, well-designed PowerPoint slides;
  do not read from the slides.
* Results (25%): clear results, and why they matter for the study of human movement.

**The honesty rule.** If the test set is used to choose a model, features or
settings, or is scored several times and the best result is kept, you lose most of
the *Model training* points. A careful result of "gait barely beats speed here" can
earn full marks. A high number from a leaky analysis cannot.